### A Demo RAG Pipeline 
##### Integrates RNGD Embedding generation/querying, reranking, and chat generation
---
This notebook demonstrate a simple RAG pileline using 
- `furiosa-ai/Qwen3-Embedding-8B` for embedding generation
- `furiosa-ai/Qwen2.5-0.5B-Instruct` for text generation 
- along with PostgreSQL and pgvector as a vector DB.

The pileline can optionally use `furiosa-ai/Qwen3-Reranker-8B` as a reranker.

It focuses on demonstration the integration points between the RNGD-hosted models (with a vLLM liek nterface and the application. 

It does not focus on optimization methods for Approximate Nearest Neighbor search.

![RAG flow](rag-flow.png)

In [4]:
# check the venv
import sys
print(sys.executable)

/home/furiosa/vertical/.venv/bin/python


In [5]:
#%pip install openai
import os
os.environ.pop("PIP_EXTRA_INDEX_URL", None)
! pip install --index-url https://pypi.org/simple/ openai


Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [6]:
# ensure ports are free, run once
from utils import launch_furiosa_server
launch_furiosa_server(name="embed", model="furiosa-ai/Qwen3-Embedding-8B", port=8001, devices="npu:1")
launch_furiosa_server(name="chat",  model="furiosa-ai/Qwen2.5-0.5B-Instruct", port=8000, devices="npu:0")
launch_furiosa_server(name="rerank", model="furiosa-ai/Qwen3-Reranker-8B", port=8002, devices="npu:2")



Started embed (eb3f32f905a2) on port 8001
embed ready at http://localhost:8001
Started chat (12b1c5e8faa3) on port 8000
chat ready at http://localhost:8000
Started rerank (2d0fbb1e3561) on port 8002
rerank ready at http://localhost:8002


'2d0fbb1e3561185cb823f23da884479362f385989040b1189620e652eabf116f'

#### Setup the embedding server at port 8001
```
docker run -it --rm \
  --device /dev/rngd:/dev/rngd \
  --security-opt seccomp=unconfined \
  --env HF_TOKEN=$HF_TOKEN \
  -v $HOME/.cache/huggingface:/root/.cache/huggingface \
  -p 8001:8001 \
  furiosaai/furiosa-llm:latest \
  serve furiosa-ai/Qwen3-Embedding-8B \
  --port 8001 \
  --devices npu:1
  ```

#### Setup the chat server at port 8000
```
docker run -it --rm \
  --device /dev/rngd:/dev/rngd \
  --security-opt seccomp=unconfined \
  --env HF_TOKEN=$HF_TOKEN \
  -v $HOME/.cache/huggingface:/root/.cache/huggingface \
  -p 8000:8000 \
  furiosaai/furiosa-llm:latest \
  serve furiosa-ai/Qwen2.5-0.5B-Instruct \
  --devices npu:0 &
```

#### Setup the reranking server at port 8000
```
docker run -it --rm \
  --device /dev/rngd:/dev/rngd \
  --security-opt seccomp=unconfined \
  --env HF_TOKEN=$HF_TOKEN \
  -v $HOME/.cache/huggingface:/root/.cache/huggingface \
  -p 8002:8002 \
  furiosaai/furiosa-llm:latest \
  serve furiosa-ai/Qwen3-Reranker-8B \
  --port 8002 \
  --devices npu:2
```

In [7]:
import os
from openai import OpenAI

### Sanity check chat server operation

In [8]:

base_url = os.getenv("OPENAI_BASE_URL", "http://localhost:8000/v1")
api_key = os.getenv("OPENAI_API_KEY", "EMPTY")

chat_client = OpenAI(api_key=api_key, base_url=base_url)

def run():
    response = chat_client.chat.completions.create(
        model="EMPTY",
        messages=[{"role": "user", "content": "What is the capital of France?"}]
    )
    return response.choices[0].message.content

print(run())


The capital of France is Paris.


### Sanity check embedding server operation

In [9]:

# Start server with: furiosa-llm serve path/to/embedding/model

base_url = os.getenv("OPENAI_BASE_URL", "http://localhost:8001/v1")
api_key = os.getenv("OPENAI_API_KEY", "EMPTY")

embeddings_client = OpenAI(base_url=base_url, api_key=api_key)

response = embeddings_client.embeddings.create(
    model="embedding-model",
    input=["Text 1", "Text 2", "Text 3"],
)
# checkout the length, you'll need it later for the vector db
for data in response.data:
    embedding = data.embedding
    print(f"Index {data.index}: {len(embedding)} dimensions")

Index 0: 4096 dimensions
Index 1: 4096 dimensions
Index 2: 4096 dimensions


### Sanity check reranking server operation

In [10]:
! pip install requests


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


OpenAI does not have a dedicated, native /v1/rerank API endpoint in its official API specification...just use post

In [11]:
import os
import requests

# Start server with: furiosa-llm serve path/to/embedding/model

reranking_base_url = os.getenv("OPENAI_BASE_URL", "http://localhost:8002/v1")
reranking_api_key = os.getenv("OPENAI_API_KEY", "EMPTY")

# Basic reranking
response = requests.post(
    f"{reranking_base_url}/rerank",
    json={
        "model": "reranker",
        "query": "What is deep learning?",
        "documents": [
            "Deep learning is a subset of machine learning using neural networks.",
            "Python is a popular programming language for data science.",
            "Machine learning algorithms learn patterns from data.",
            "Neural networks are inspired by biological neural networks.",
            "JavaScript is used for web development.",
        ],
    },
)

data = response.json()
print(f"Model: {data['model']}")
print(f"Total results: {len(data['results'])}")
print()

# Results are sorted by relevance (descending)
for result in data["results"]:
    print(f"Rank {result['index'] + 1}: score = {result['relevance_score']:.4f}")
    print(f"  Document: {result['document']['text'][:60]}...")
    print()


Model: furiosa-ai/Qwen3-Reranker-8B
Total results: 5

Rank 1: score = 0.9855
  Document: Deep learning is a subset of machine learning using neural n...

Rank 4: score = 0.1441
  Document: Neural networks are inspired by biological neural networks....

Rank 3: score = 0.0770
  Document: Machine learning algorithms learn patterns from data....

Rank 2: score = 0.0001
  Document: Python is a popular programming language for data science....

Rank 5: score = 0.0000
  Document: JavaScript is used for web development....



### Setup PostgreSQL and pgvector db

In [12]:

# https://hub.docker.com/r/ramsrib/pgvector
!docker stop langfuse-postgres-1  # may be required since it runs on the same port
!docker rm -f pgvector-16 2>/dev/null; docker run -d --name pgvector-16 -p 5432:5432 -e POSTGRES_PASSWORD=postgres ramsrib/pgvector:16


langfuse-postgres-1
pgvector-16
65279eb2142199039841b70cb38dd225bfbdf19ea69c653c1aa2f313431deb6e


### Setup the psycopg for Python-based interaction with the db

In [13]:

! pip install "psycopg[binary]" pgvector


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


### Create a Table for Embeddings plus the corresponding text (te be retrieved)
Create a table that stores text documents along with their embeddings:

In [14]:
import time
import psycopg

DSN = "host=localhost port=5432 dbname=postgres user=postgres password=postgres"

conn = None
for attempt in range(30):
    try:
        conn = psycopg.connect(DSN, autocommit=True, connect_timeout=2)
        print(f"Postgres ready after {attempt + 1} attempt(s)")
        break
    except psycopg.OperationalError:
        time.sleep(1)
if conn is None:
    raise RuntimeError("Postgres did not become ready in 30s")

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            id SERIAL PRIMARY KEY,
            content TEXT,
            embedding VECTOR(4096)
        );
    """)

print("Table created.")

Postgres ready after 2 attempt(s)
Table created.


### Insert Embeddings with the psycopq library
You can generate embeddings in Python using OpenAI or any other embedding model and insert them into PostgreSQL.

In [15]:
import psycopg
from pgvector.psycopg import register_vector
from openai import OpenAI

conn = psycopg.connect(
    "host=localhost port=5432 dbname=postgres user=postgres password=postgres",
    autocommit=True,
)
register_vector(conn)

texts = [
    "PostgreSQL is an open-source relational database.",
    "RAG combines retrieval and generation for better answers.",
    "pgvector adds vector search capabilities to Postgres.",
    "PostgreSQL scales well vertically (up to ~10-100 TB) by increasing CPU/RAM, but it hits bottlenecks at massive scale.",
    "The scalability of PostgreSQL with pgvector is generally effective for datasets up to tens of millions of vectors.",
    "quote - I tested pgvector on 5 million 512-dimensional vectors, and the performance was good enough with 1 sec similarity search.",
]

response = embeddings_client.embeddings.create(
    model="embedding-model",
    input=texts,
)

with conn.cursor() as cur:
    for text, data in zip(texts, response.data):
        cur.execute(
            "INSERT INTO documents (content, embedding) VALUES (%s, %s)",
            (text, data.embedding),
        )

print(f"Inserted {len(texts)} documents.")

Inserted 6 documents.


### Query and Retrieve (semantically)
To find similar documents, you embed the user query and perform a similarity search using PostgreSQL’s vector operators.
In reality, this would be a hybrid retrieval, not just pure semantic retrieval

In [16]:
query = "How do I search vectors in Postgres?"

query_embedding = embeddings_client.embeddings.create(
    input=query,
    model="embedding-model",
).data[0].embedding

with conn.cursor() as cur:
    cur.execute("""
        SELECT content, embedding <-> %s::vector AS distance
        FROM documents
        ORDER BY distance ASC
        LIMIT 4;
    """, (query_embedding,))
    results = cur.fetchall()

for content, distance in results:
    print(f"{distance:.4f}  {content}")

0.5272  pgvector adds vector search capabilities to Postgres.
0.7335  quote - I tested pgvector on 5 million 512-dimensional vectors, and the performance was good enough with 1 sec similarity search.
0.8439  The scalability of PostgreSQL with pgvector is generally effective for datasets up to tens of millions of vectors.
0.9459  PostgreSQL is an open-source relational database.


### Rerank what was retrieved

In [17]:
response = requests.post(
    f"{reranking_base_url}/rerank",
    json={
        "model": "reranker",
        "query": query,
        "documents": [content for content, _ in results],
    },
)

reranked_results = [
    (r["document"]["text"], r["relevance_score"])
    for r in response.json()["results"]
]

print(f"Query: {query}\n")
print(f"Reranked {len(reranked_results)} documents:\n")
for rank, (content, score) in enumerate(reranked_results, start=1):
    print(f"  [{rank}] score={score:.4f}")
    print(f"      {content}")
    print()



Query: How do I search vectors in Postgres?

Reranked 4 documents:

  [1] score=0.9230
      pgvector adds vector search capabilities to Postgres.

  [2] score=0.6298
      The scalability of PostgreSQL with pgvector is generally effective for datasets up to tens of millions of vectors.

  [3] score=0.5600
      quote - I tested pgvector on 5 million 512-dimensional vectors, and the performance was good enough with 1 sec similarity search.

  [4] score=0.0434
      PostgreSQL is an open-source relational database.



### Generate a cogent response
Once you retrieve the most similar text snippets, you can feed them into an LLM as context to produce grounded responses.

In [18]:
print(f"-query:\n{query}")
context = "\n".join(content for content, _ in reranked_results)
print(f"-context:\n{context}")
prompt = f"""You are an assistant that answers questions ONLY using the context below. Do not use any other information. If you don't find anything relevant in the context, just say "I don't know".

Context:
{context}

Question: {query}
"""

response = chat_client.chat.completions.create(
    model="EMPTY",
    messages=[{"role": "user", "content": prompt}],
)

print(f"-response:\n{response.choices[0].message.content}")

-query:
How do I search vectors in Postgres?
-context:
pgvector adds vector search capabilities to Postgres.
The scalability of PostgreSQL with pgvector is generally effective for datasets up to tens of millions of vectors.
quote - I tested pgvector on 5 million 512-dimensional vectors, and the performance was good enough with 1 sec similarity search.
PostgreSQL is an open-source relational database.
-response:
To search vectors in PostgreSQL using pgvector, you can follow these steps:

1. **Install pgvector:** If you haven't already, you'll need to install pgvector in your PostgreSQL environment. You can install it through the official website or by installing it via package managers like `pip`.

2. **Set up indexing for your vectors:** Ensure that your vectors are well-indexed and that your Postgres database supports efficient vector operations such as spatial and vector joins.

3. **Constructor queries:** Use the `pgvector::vector` function in SQL to create and manipulate vectors. T

In [19]:
from utils import stop_server
stop_server("embed")
stop_server("chat")
stop_server("rerank")

Stopped embed
Stopped chat
Stopped rerank


### Credits

This notebook is based on the use of PostgreSQL from the link below.

https://medium.com/@nitinprodduturi/using-postgresql-as-a-vector-database-for-rag-retrieval-augmented-generation-c62cfebd9560